# Regio-AI: Stage 1 — Satellite Data Exploration & Strandskydd Visualization

**Project:** Fine-tuning IBM/NASA Prithvi for Strandskydd Violation Detection  
**Platform:** OpenShift AI — PyTorch notebook (CPU)  
**Stage:** 1 of N — Data Exploration & Environment Setup  

---

## Background

Sweden's *strandskydd* (shoreline protection law) prohibits construction within **100 metres of any water body** — lakes, rivers, and the sea. Enforcing this law over the entire Swedish coastline and inland water network is a significant challenge for municipalities and county boards (*länsstyrelserna*).

This project explores using satellite imagery and the **IBM/NASA Prithvi geospatial foundation model** to automatically detect buildings that may violate strandskydd — flagging potential cases for human review.

## Stage 1 Goals

1. Set up the geospatial Python environment
2. Connect to Digital Earth Sweden (OpenEO) and explore available data
3. Define a representative Area of Interest (AOI)
4. Visualise the strandskydd protection zone interactively on a map
5. Lay the groundwork for Stage 2 (building detection with Prithvi)

---

## 1. Environment Setup

Install required geospatial and ML-adjacent packages. This cell only needs to run once per notebook server session.

In [ ]:
%pip install --quiet openeo rasterio geopandas shapely folium owslib requests numpy matplotlib pillow pyproj
print("Installation complete.")

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import openeo
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import folium
from folium import plugins
from shapely.geometry import box, Point, mapping
import pyproj
from pyproj import Transformer
import requests

print(f"openeo      : {openeo.__version__}")
print(f"geopandas   : {gpd.__version__}")
print(f"folium      : {folium.__version__}")
print("All imports successful.")

## 3. Configuration — Area of Interest & Parameters

We focus on **Värmdö municipality** in the Stockholm Archipelago — an area well known for strandskydd discussions due to its dense mix of water, islands, and summer houses.

All coordinates are in **WGS84 (EPSG:4326)**.

In [ ]:
# --- Area of Interest ---
AOI_NAME = "Värmdö, Stockholm Archipelago"

AOI_BBOX = {
    "west":  18.40,
    "south": 59.25,
    "east":  18.65,
    "north": 59.40
}

AOI_CENTER = [
    (AOI_BBOX["south"] + AOI_BBOX["north"]) / 2,
    (AOI_BBOX["west"]  + AOI_BBOX["east"])  / 2
]

# --- Strandskydd ---
STRANDSKYDD_DISTANCE_M = 100  # metres from water (standard zone)

# --- Sentinel-2 parameters (used in Stage 2 with authentication) ---
TEMPORAL_EXTENT = ["2023-06-01", "2023-09-30"]  # Summer — fewer clouds
BANDS           = ["B02", "B03", "B04", "B08"]   # Blue, Green, Red, NIR
MAX_CLOUD_COVER = 20                              # %

print(f"AOI          : {AOI_NAME}")
print(f"Bounding box : {AOI_BBOX}")
print(f"Center       : {AOI_CENTER}")
print(f"Strandskydd  : {STRANDSKYDD_DISTANCE_M} m from water")
print(f"Temporal     : {TEMPORAL_EXTENT}")
print(f"Bands        : {BANDS}")

## 4. Digital Earth Sweden — OpenEO Connection

[Digital Earth Sweden (DES)](https://digitalearth.se) provides access to Sentinel-2 satellite data over Sweden via the OpenEO standard API.

**Note on authentication:**  
Browsing metadata (collections, capabilities) works anonymously. Downloading actual datacubes requires a DES account. Register at [digitalearth.se](https://digitalearth.se) to get credentials — the registration is free for research and public-sector use.

```python
# Once you have credentials, authenticate with:
conn.authenticate_oidc()  # Browser-based OIDC login
# or
conn.authenticate_basic(username="your@email.se", password="...")
```

In [ ]:
OPENEO_URL = "https://openeo.digitalearth.se"

try:
    conn = openeo.connect(OPENEO_URL)
    caps = conn.capabilities()
    print(f"Connected    : {OPENEO_URL}")
    print(f"API version  : {caps.api_version()}")
    print(f"Backend      : {caps.get('title', 'Digital Earth Sweden')}")
except Exception as e:
    conn = None
    print(f"Could not connect to {OPENEO_URL}")
    print(f"Error        : {e}")
    print("Continuing with public tile visualisation.")

In [ ]:
# Browse available collections (anonymous metadata access)
if conn:
    try:
        collections = conn.list_collections()
        print(f"Available collections: {len(collections)}\n")
        for c in sorted(collections, key=lambda x: x['id']):
            title = c.get('title', '—')
            print(f"  {c['id']:<35} {title}")
    except Exception as e:
        print(f"Could not list collections: {e}")
else:
    print("Skipped — no active connection.")

## 5. Sentinel-2 Datacube Definition

The cell below defines how the Sentinel-2 datacube will be loaded from DES once authentication is in place. This is the primary data input for Prithvi inference in Stage 2.

For the IBM/NASA Prithvi model, we need to produce **HLS-compatible** 6-band input:

| Prithvi band | Sentinel-2 band | Description |
|---|---|---|
| 1 | B02 | Blue |
| 2 | B03 | Green |
| 3 | B04 | Red |
| 4 | B8A | Narrow NIR |
| 5 | B11 | SWIR 1 |
| 6 | B12 | SWIR 2 |

In [ ]:
# Prithvi expects 6-band HLS-compatible input
PRITHVI_BANDS = ["B02", "B03", "B04", "B8A", "B11", "B12"]

if conn:
    try:
        # Uncomment after authenticating with conn.authenticate_oidc()
        # datacube = conn.load_collection(
        #     collection_id="sentinel-2-l2a",
        #     spatial_extent=AOI_BBOX,
        #     temporal_extent=TEMPORAL_EXTENT,
        #     bands=PRITHVI_BANDS,
        #     max_cloud_cover=MAX_CLOUD_COVER
        # )
        # print(f"Datacube     : {datacube}")
        print("Datacube definition ready.")
        print(f"Bands        : {PRITHVI_BANDS}  (HLS-compatible for Prithvi)")
        print(f"Temporal     : {TEMPORAL_EXTENT}")
        print(f"Cloud cover  : <= {MAX_CLOUD_COVER}%")
        print("\n>>> Authenticate with conn.authenticate_oidc() to activate data download.")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipped — no active connection.")

## 6. Strandskydd Visualisation

The interactive map below combines:

- **Satellite imagery** — Esri World Imagery (public tile service)
- **OpenStreetMap** — reference layer
- **Area of Interest** — the Värmdö bounding box
- **Strandskydd zones** — WMS layer from [Naturvårdsverket](https://www.naturvardsverket.se) (Swedish EPA), the authoritative public source

The Naturvårdsverket publishes strandskydd boundaries as an open WMS service under the Swedish INSPIRE obligations.

In [ ]:
# --- Create base map ---
m = folium.Map(
    location=AOI_CENTER,
    zoom_start=12,
    tiles=None,
    control_scale=True
)

# --- Base layers ---
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Esri World Imagery",
    name="Satellite (Esri)",
    overlay=False,
    control=True
).add_to(m)

folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap",
    overlay=False,
    control=True
).add_to(m)

# --- Area of Interest bounding box ---
folium.Rectangle(
    bounds=[
        [AOI_BBOX["south"], AOI_BBOX["west"]],
        [AOI_BBOX["north"], AOI_BBOX["east"]]
    ],
    color="#FFD700",
    weight=2,
    fill=True,
    fill_color="#FFD700",
    fill_opacity=0.05,
    tooltip=f"Area of Interest: {AOI_NAME}",
    name="Area of Interest"
).add_to(m)

# --- Strandskydd WMS from Naturvårdsverket ---
# Source: https://nvpub.vic-metria.nu — Swedish EPA ArcGIS MapServer
# Published under Swedish INSPIRE obligations (open access)
strandskydd_wms_url = (
    "https://nvpub.vic-metria.nu/arcgis/services/"
    "Strandskydd/MapServer/WmsServer"
)

folium.WmsTileLayer(
    url=strandskydd_wms_url,
    name="Strandskydd (Naturvardsverket)",
    layers="0",
    fmt="image/png",
    transparent=True,
    overlay=True,
    control=True,
    opacity=0.65,
    attr="Naturvardsverket"
).add_to(m)

# --- Layer control & minimap ---
folium.LayerControl(collapsed=False).add_to(m)
plugins.MiniMap(toggle_display=True).add_to(m)
plugins.Fullscreen().add_to(m)

# --- Render ---
print(f"Map centre   : {AOI_CENTER}")
print(f"AOI          : {AOI_NAME}")
print(f"Strandskydd  : {strandskydd_wms_url}")
m

## 7. NDWI Water Detection — Preview for Stage 2

Once Sentinel-2 data is downloaded, we can compute the **Normalised Difference Water Index (NDWI)** to automatically extract water bodies and derive the strandskydd 100m buffer zone programmatically.

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$

Pixels with NDWI > 0 are classified as water. The strandskydd zone is then a spatial buffer around those pixels.

This approach means we are **not dependent on pre-existing strandskydd boundary data** — we derive the protection zone directly from the satellite signal.

In [ ]:
def compute_ndwi(green_band: np.ndarray, nir_band: np.ndarray) -> np.ndarray:
    """
    Compute Normalised Difference Water Index.

    Parameters
    ----------
    green_band : np.ndarray  (Sentinel-2 B03)
    nir_band   : np.ndarray  (Sentinel-2 B08 or B8A)

    Returns
    -------
    ndwi : np.ndarray  values in [-1, 1], >0 indicates water
    """
    green = green_band.astype(float)
    nir   = nir_band.astype(float)
    denom = green + nir
    denom[denom == 0] = np.nan  # avoid divide-by-zero
    return (green - nir) / denom


# --- Demonstrate on synthetic data ---
rng   = np.random.default_rng(42)
green = rng.integers(500, 3000, size=(256, 256), dtype=np.uint16)
nir   = rng.integers(200, 4000, size=(256, 256), dtype=np.uint16)

# Simulate a water body in the top-left quadrant
green[:128, :128] = rng.integers(2000, 3000, size=(128, 128))
nir[:128, :128]   = rng.integers(100,   600, size=(128, 128))

ndwi       = compute_ndwi(green, nir)
water_mask = ndwi > 0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(green, cmap="Greens")
axes[0].set_title("Green band (B03)")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(ndwi, cmap="RdYlBu", vmin=-1, vmax=1)
axes[1].set_title("NDWI")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(water_mask, cmap="Blues")
axes[2].set_title(f"Water mask (NDWI > 0)\n{water_mask.sum():,} px ({water_mask.mean()*100:.1f}%)")
plt.colorbar(im2, ax=axes[2])

plt.suptitle("NDWI Pipeline — Synthetic Demo (Stage 2 will use real Sentinel-2 tiles)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("NDWI demo complete.")

## 8. Summary & Next Steps

### Stage 1 complete

| Step | Status |
|---|---|
| Geospatial environment | Ready |
| OpenEO connection to Digital Earth Sweden | Ready (auth needed for data download) |
| Area of Interest defined | Värmdö, Stockholm Archipelago |
| Strandskydd visualisation | WMS overlay rendered |
| NDWI pipeline | Demonstrated on synthetic data |

### Stage 2 — Building Detection with Prithvi

1. Authenticate with Digital Earth Sweden and download a Sentinel-2 tile over the AOI
2. Preprocess the 6-band HLS-compatible input for the Prithvi model
3. Load `ibm-nasa-geospatial/Prithvi-100M` from HuggingFace
4. Attach a lightweight segmentation head and run inference (CPU)
5. Compute NDWI water mask → derive strandskydd 100m buffer
6. Overlay detected buildings against the buffer to flag potential violations

### Resources

- [Digital Earth Sweden](https://digitalearth.se) — data platform & OpenEO docs
- [Prithvi on HuggingFace](https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M)
- [Naturvårdsverket geodata](https://www.naturvardsverket.se/verktyg-och-tjanster/kartor-och-geografisk-information/) — authoritative strandskydd boundaries
- [Lantmäteriet](https://www.lantmateriet.se) — building footprint data (TopoDB)
- [OpenEO Python client docs](https://open-eo.github.io/openeo-python-client/)